<br>
<left><bold><font size="5"> Customer Churn Prediction and Segmentation for Telecom</font></bold></left>
<br>
<br>


## Table of contents
* [1. Overview and Goals](#introduction)
* [2. Data Source and Use Case](#data)
* [3. ETL (Extract Transform Load)](#etl)
* [4. Analysis & Visualization](#Analysis&Visualization)
* [5. Model Training and Evaluation](#Modeling)
* [6. Customer Segmentation](#Segmentation)
* [7. Results and Discussion](#results)

## 1. Overview and Goals <a name="introduction"></a>

Customer acquisition and **retention** is a key concern for many industries, especially acute in the strongly competitive and quick growth **telecommunications** industry. Meanwhile, since the **cost** of retaining a good customer is much lower than acquiring a new one, it is very profit-effective to input valuable resource on the Retention Campaign.

Customers churn for various different reasons. Experience tells us that once the customer has made up their mind, retention becomes really hard. Therefore, managing churn in a **proactive** process is essential for the growth of the customer base and profit.

The primary goal of churn analysis is usually to create a list of contracts that are likely to be cancelled in the near future. 

The customers holding these contracts are then targeted with special incentives designed to deter cancellation.

At a more sophisticated level, the telecommunications company attempts to detect the reasons for an expected cancellation, because this information may help customize the offer.

<img style="float: left;" src='https://parcusgroup.com/public/wysiwyg/images/Telecom_Customer_Churn_Prediction_Models.jpg' width='50%'>


## 2. Data Source and Use Case <a name="data"></a>

The Data Source I've chosen here is an open source data(cell2cell) by Teradata center for customer relationship management at Duke University.

Cell2Cell dataset is preprocessed and a balanced version provided for analyzing Process. consists of 71,047 instances and 58 attributes. We can try with data set, try Machine learning algorithms and deep leaning algorithms find measures like accuracy, ROC, AUC

Telecom Industry faces fierce com-petition in satisfying its customers. The role of churn prediction system is not only restricted to accurately predict churners but also to interpret customer churn behavior.

Churn management in the telecom services industry is used to securing the customers for a company. In essence, proper customer management presumes an ability to forecast the customer decides to move to another service provider, a measurement of customer profitability, and a set of strategic and tactic retention measures to reduce the movement.

In practice, we can segment its customers by profitability and focus retention management only on those profitable segments, and score the entire customer base with propensity to churn and prioritize the retention effort based on profitability and churn propensity.

The origin link of the dataset is https://www.kaggle.com/jpacse/datasets-for-churn-telecom


In [ ]:
# Check the version of pyspark
# Pyspark Version should be: 2.4.0
!pyspark --version


In [ ]:
# Import pyspark
import pyspark
from pyspark import SparkConf
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, TrainValidationSplit
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.linalg import Vectors
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator, ClusteringEvaluator

from pyspark.sql.types import FloatType, StringType, DoubleType
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# Import numpy, pandas, matpoltlib, seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, classification_report
from sklearn import metrics

# import xgboost
import xgboost as xgb

# Import keras
import keras
from keras import Sequential, regularizers
from keras.optimizers import SGD, RMSprop, Adam
from keras.layers import Dense, Dropout, Activation
from keras.wrappers.scikit_learn import KerasRegressor
from keras.utils import np_utils

sns.set_style("whitegrid")  # Set style of seaborn


In [ ]:
# Create SparkSession in local environment
spark = (
    SparkSession.builder.appName("Telecom_Churn")  # create a name
    .config("spark.executor.memory", "10g")  # Maximum amount of memory allowed
    .config("spark.executor.cores", "3")  # Maximum cpu threads allowed
    .config("spark.cores.max", "3")
    .config("spark.driver.memory", "10g")
    .config("spark.sql.broadcastTimeout", "9000")
    .getOrCreate()
)
SparkConf().getAll()


In [ ]:
# Define a load founction
def load(path, show_num=None, printable=True):
    """
    read csv file, drop null values, print the length of the rows
    """
    data = spark.read.csv(path, header="true", inferSchema=True)
    data = data.na.drop()
    if show_num:
        data.show(show_num)

    if printable == True:
        data.printSchema()
        print("Length:", data.count())
    return data


In [ ]:
# Define the path of data source and load into spark dataframe
path = "../input/cell2celltrain.csv"
df = load(path)
df.select("MonthlyRevenue").describe().show()


## 3. ETL (Extract Transform Load) <a name="etl"></a>

Before Modeling and Visualization, first of all we need follow the data engineering process, specificlly ETL process, which stands for Extract, transform, and load variables from identified data items.

Extract is the process of reading data from a database. In this stage, the data is collected, often from multiple and different types of sources.

Transform is the process of converting the extracted data from its previous form into the form it needs to be in so that it can be placed into another database. Transformation occurs by using rules or lookup tables or by combining the data with other data.

Load is the process of writing the data into the target database.

<img style="float: left;" src='https://docs.microsoft.com/en-us/azure/architecture/data-guide/images/etl.png' width='50%'>
<br>

In [ ]:
# Delete the rows with 'NA' values
for column in df.columns:
    if df.schema[column].dataType == pyspark.sql.types.StringType():
        df = df.filter(df[column] != "NA")


In [ ]:
# Check the description of Churn column
df.select("Churn").describe().show()


In [ ]:
# Going through all columns required to be transfer into numerical values
columns_string = [
    column
    for column in df.columns
    if df.schema[column].dataType == pyspark.sql.types.StringType()
]
columns_string


In [ ]:
# create the view of df under spark
df.createOrReplaceTempView("df")

# check the distribution of picked columns
spark.sql(
    "SELECT \
            Churn, \
            CAST (avg(MonthlyRevenue) as decimal(8,2)) as avg_MonthlyRevenue, \
            CAST (avg(MonthlyMinutes) as decimal(8,2)) as avg_MonthlyMinutes, \
            CAST (avg(CurrentEquipmentDays) as decimal(8,2)) as avg_CurrentEquipmentDays, \
            CAST (avg(TotalRecurringCharge) as decimal(8,2)) as avg_TotalRecurringCharge, \
            CAST (avg(OverageMinutes) as decimal(8,2)) as avg_OverageMinutes, \
            CAST (avg(RoamingCalls) as decimal(8,2)) as avg_RoamingCalls \
            FROM df GROUP BY Churn"
).show()


In [ ]:
# Check the values of string columns
df.select([column for column in df.columns if column in columns_string]).show(1)


In [ ]:
# Based on the values of the string columns, select the list of columns for StringIndexer
columns_for_indexer = [
    "Churn",
    "ChildrenInHH",
    "HandsetRefurbished",
    "HandsetWebCapable",
    "TruckOwner",
    "RVOwner",
    "Homeownership",
    "BuysViaMailOrder",
    "RespondsToMailOffers",
    "OptOutMailings",
    "NonUSTravel",
    "OwnsComputer",
    "HasCreditCard",
    "NewCellphoneUser",
    "NotNewCellphoneUser",
    "OwnsMotorcycle",
    "MadeCallToRetentionTeam",
    "MaritalStatus",
]


In [ ]:
# Loop the list and use StringIndexer encodes the string columns of labels(Yes or No) to columns of label indices(1 or 0)
for column in columns_for_indexer:
    indexer = StringIndexer(inputCol=column, outputCol=column + "Index")
    df = indexer.fit(df).transform(df)


In [ ]:
# Drop the original string columns of labels
df = df.select([column for column in df.columns if column not in columns_for_indexer])


In [ ]:
# Check the Churn column again to confirm the result
df.select("ChurnIndex").describe().show()


In [ ]:
# Replace unknown values to 0 for Price Column
df = df.replace(to_replace={"Unknown": "0"}, subset=["HandsetPrice"])


In [ ]:
# Delete the rows with 'NA' values
for column in df.columns:
    if df.schema[column].dataType == pyspark.sql.types.StringType():
        df = df.filter(df[column] != "Unknown")


In [ ]:
# Another Iteration for feature engineering
columns_string = [
    column
    for column in df.columns
    if df.schema[column].dataType == pyspark.sql.types.StringType()
]
columns_string


In [ ]:
# Mapping string values into integer based on its values
mapping_PrizmCode = {"Other": "0", "Suburban": "1", "Town": "2", "Rural": "3"}
df = df.replace(to_replace=mapping_PrizmCode, subset=["PrizmCode"])


In [ ]:
# Prepare the list of columns to be transformed into float
columns_to_float = [
    "MonthlyRevenue",
    "MonthlyMinutes",
    "TotalRecurringCharge",
    "DirectorAssistedCalls",
    "OverageMinutes",
    "RoamingCalls",
    "PercChangeMinutes",
    "PercChangeRevenues",
]


In [ ]:
# Transform ServiceArea and CreditRating columns only to keep the int value
df = df.withColumn("ServiceArea", df["ServiceArea"].substr(-3, 3))
df = df.withColumn("CreditRating", df["CreditRating"].substr(1, 1))


In [ ]:
# Transform the type of columns to float
for column in columns_to_float:
    df = df.withColumn(column, df[column].cast("float"))


In [ ]:
# Create a Pandas DataFrame for data visualization
df_pd = df.toPandas()


In [ ]:
# Before transform the type of Occupation column, let's check it's distribution first
graph = df_pd[["Occupation", "MonthlyRevenue"]]
ax = sns.barplot(x="Occupation", y="MonthlyRevenue", data=graph)


In [ ]:
# Transform string values to numbers using mapping
temp = (
    df_pd.loc[:, ["Occupation", "MonthlyRevenue"]]
    .groupby("Occupation")
    .mean()
    .sort_values(["MonthlyRevenue"], ascending=[0])
)

mapping_Occupation = dict([temp.index[i], str(i)] for i in range(len(temp)))
print(mapping_Occupation)

df = df.replace(to_replace=mapping_Occupation, subset=["Occupation"])


In [ ]:
df.describe()


In [ ]:
# Prepare the list of columns to be transformed into integer
columns_to_int = [
    "Handsets",
    "HandsetModels",
    "CurrentEquipmentDays",
    "AgeHH1",
    "AgeHH2",
    "HandsetPrice",
    "ServiceArea",
    "CreditRating",
    "PrizmCode",
    "Occupation",
    "ChurnIndex",
    "ChildrenInHHIndex",
    "HandsetRefurbishedIndex",
    "HandsetWebCapableIndex",
    "TruckOwnerIndex",
    "RVOwnerIndex",
    "HomeownershipIndex",
    "BuysViaMailOrderIndex",
    "RespondsToMailOffersIndex",
    "OptOutMailingsIndex",
    "NonUSTravelIndex",
    "OwnsComputerIndex",
    "HasCreditCardIndex",
    "NewCellphoneUserIndex",
    "NotNewCellphoneUserIndex",
    "OwnsMotorcycleIndex",
    "MadeCallToRetentionTeamIndex",
    "MaritalStatusIndex",
]


In [ ]:
# Transform the type of columns to integer
for column in columns_to_int:
    df = df.withColumn(column, df[column].cast("int"))


## 4. Analysis & Visualization <a name="Analysis&Visualization"></a>

Before go to next step, I'd like to do some Analysis & Visualization of the features we just cleaned. So we can have better understanding on the business aspect.

Based on the industry experience, usually there're five dimensions and listed below are the significant variables.
-	Customer demography: Age, Tenure, Gender, Location, Zip code, etc.
-	Bill and payment: Monthly fee, Billing amount, Count of overdue payment, payment method, Billing type, etc.
-	Network/TV/Phone usage records: Network use frequency, network flow, usage time and period, internet average speed, In-net call duration, Call type, etc.
-	Customer care/service: Service call number, service type, service duration, account change count
-	Competitors information: Offer detail under similar plan, etc

As we can see based on our original database, we're lack of several features such as competitors information. In the real business projects I experienced that is extremely important, however since we're only showing technical process here, I'll just leave it.

In [ ]:
# Go through the dataset check the distribution of each feature
for i in range(len(df.columns) // 6 + 1):
    df.describe(df.columns[6 * i : 6 * (i + 1)]).show()


In [ ]:
# Create a new Pandas DataFrame for data visualization
df_pd = df.toPandas()


In [ ]:
# Define Catplot function and Barplot function
def cat_plot(feature, cut=12):
    temp = df_pd.loc[:, [feature, "ChurnIndex"]]
    temp[feature + "_binned"] = pd.qcut(temp[feature], cut, duplicates="drop")
    ax = sns.catplot(
        x="ChurnIndex",
        y=feature + "_binned",
        data=temp,
        kind="bar",
        height=5,
        aspect=2.7,
    )


def bar_plot(feature, cut=False, logscale=False, drop_zero=False):
    if drop_zero:
        temp = df_pd[df_pd[feature] != 0].loc[:, [feature, "ChurnIndex"]]
    else:
        temp = df_pd.loc[:, [feature, "ChurnIndex"]]

    if cut > 0:
        temp[feature + "_binned"] = pd.qcut(temp[feature], cut, duplicates="drop")
        ax = sns.barplot(x=feature + "_binned", y="ChurnIndex", data=temp)
    else:
        ax = sns.barplot(x=feature, y="ChurnIndex", data=temp)

    if logscale:
        ax.set_yscale("log")


In [ ]:
# Use defined Catplot function plot the distribution of MonthsInService
cat_plot("MonthsInService", cut=12)


In [ ]:
# Use defined Barplot function plot the distribution of MonthlyMinutes
bar_plot("MonthlyMinutes", cut=5, logscale=True, drop_zero=True)


In [ ]:
# Use defined Barplot function plot the distribution of OverageMinutes
bar_plot("OverageMinutes", cut=7, logscale=True, drop_zero=True)


In [ ]:
# Use defined Barplot function plot the distribution of CreditRating
bar_plot("CreditRating", cut=5, logscale=True)


In [ ]:
# Use defined Barplot function plot the distribution of AgeHH1
bar_plot("AgeHH1", cut=6, logscale=True, drop_zero=True)


In [ ]:
# Subplots the distribution of HandsetRefurbished, HandsetWebCapable and MadeCallToRetentionTeam into one graph
feature = [
    "HandsetRefurbishedIndex",
    "HandsetWebCapableIndex",
    "MadeCallToRetentionTeamIndex",
    "ChurnIndex",
]
temp = df_pd.loc[:, feature]
temp.columns = [
    "HandsetRefurbished",
    "HandsetWebCapable",
    "MadeCallToRetentionTeam",
    "ChurnIndex",
]

fig, axs = plt.subplots(figsize=(10, 5), ncols=3)
sns.barplot(x="HandsetRefurbished", y="ChurnIndex", data=temp, ax=axs[0])
sns.barplot(x="HandsetWebCapable", y="ChurnIndex", data=temp, ax=axs[1])
sns.barplot(x="MadeCallToRetentionTeam", y="ChurnIndex", data=temp, ax=axs[2])


## 5. Prediction Model Training and Evaluation <a name="Modeling"></a>

The first step of this module is data preprocessing. We'll start from VectorAssembler, which is useful for combining raw features and features generated by different feature transformers into a single feature vector, in order to train ML models like Random Forest and Gradient-Boosted Trees. Next, since we're going to use K-Means and Neural Network for the modeling, we're going to use StandardScaler transforms the dataset of Vector rows, normalizing each feature to have unit standard deviation and/or zero mean. Last, we'll use down-samplings method to deal with our unbalance dataset.

Next Step is modeling, in this project we'll use Random Forest Classifier and GBTs Classifier to predict customer Churn; Use XGBoost Regressor and Neural Network to predict Revenue Loss.

In the last part of each modeling iteration is evaluate the performance of each prediction model, in order to do so, we will split the dataset in 2 parts: training set (80% of the entries in the dataset) and test set (20% of the entries in the dataset). We train the model on the training set and then we test the model on the test set. I decided to use the ROC curve(A receiver operating characteristic curve) to test the model performance for classification problem which is to predict customer Churn in our case; And use mean absolute error for regression program which is predicted revenue loss in our case, then I'll plot a chart of how many of total revenue loss can be covered with our predicted highest risk group.


### 5.1 Data Preprocessing

In [ ]:
# Create column list for VectorAssembler
inputcols = [
    column for column in df.columns if column not in ["CustomerID", "ChurnIndex"]
]


In [ ]:
# Use vectorAssembler transform the given list of columns into a single vector column.
vectorAssembler = VectorAssembler(inputCols=inputcols, outputCol="features")
features_vectorized = vectorAssembler.transform(df)
features_vectorized = features_vectorized.withColumnRenamed("ChurnIndex", "label")


In [ ]:
# StandardScaler
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

# Compute summary statistics by fitting the StandardScaler
scalerModel = scaler.fit(features_vectorized)

# Normalize each feature to have unit standard deviation.
scaledData = scalerModel.transform(features_vectorized)
scaledData.show()


In [ ]:
# Check the distribution of label
scaledData.select("label").groupBy("label").count().collect()


Since the data set is highly skewed – we have much more Negative training samples than Positive training samples – we will need to try out some strategies that counter the unbalance.

Here I'm going to use one of the simplest method called down-samplings, which means we just randomly filter out some of the majority cases.

In [ ]:
# Down-sampling
scaledData_ds, _ = scaledData.filter(scaledData["label"] == 0.0).randomSplit([0.4, 0.6])
scaledData_ds = scaledData.filter(scaledData["label"] == 1.0).union(scaledData_ds)
scaledData_ds.select("label").groupBy("label").count().collect()


In [ ]:
# Split dataset into train and test set
train, test = scaledData_ds.randomSplit([0.8, 0.2])


In [ ]:
# Define plot roc curve function
def plot_roc_curve(model, train, test):
    predictions = model.transform(test)
    predictions_train = model.transform(train)

    results = predictions.select(["probability", "label"])
    results_train = predictions_train.select(["probability", "label"])

    ## prepare score-label set
    results_collect = results.collect()
    results_list = [(float(i[0][0]), 1.0 - float(i[1])) for i in results_collect]

    results_collect_train = results_train.collect()
    results_list_train = [
        (float(i[0][0]), 1.0 - float(i[1])) for i in results_collect_train
    ]

    fpr = dict()
    fpr_train = dict()
    tpr = dict()
    tpr_train = dict()
    roc_auc = dict()
    roc_auc_train = dict()

    y_test = [i[1] for i in results_list]
    y_score = [i[0] for i in results_list]

    y_train = [i[1] for i in results_list_train]
    y_score_train = [i[0] for i in results_list_train]

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)

    fpr_train, tpr_train, _ = roc_curve(y_train, y_score_train)
    roc_auc_train = auc(fpr_train, tpr_train)

    plt.figure()
    plt.plot(fpr, tpr, label="ROC curve on testing set (area = %0.2f)" % roc_auc)
    plt.plot(
        fpr_train,
        tpr_train,
        label="ROC curve on training set (area = %0.2f)" % roc_auc_train,
    )
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Receiver operating characteristic example")
    plt.legend(loc="lower right")
    plt.show()


### 5.2 Random Forest Classifier

In [ ]:
# Validation for hyper-parameter tuning.
# Randomly splits the input dataset into train and validation sets,
# and uses evaluation metric on the validation set to select the best model.
rf = RandomForestClassifier(
    featuresCol="features", labelCol="label", predictionCol="prediction", maxBins=16
)

# Use a ParamGridBuilder to construct a grid of parameters to search over.
# TrainValidationSplit will try all combinations of values and determine best model using the evaluator.
paramGrid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [8, 10, 12])
    .addGrid(rf.minInstancesPerNode, [1, 3, 5, 10])
    .addGrid(rf.numTrees, [50, 100])
    .build()
)

# In this case the estimator is BinaryClassificationEvaluator
# A TrainValidationSplit requires an Estimator, a set of Estimator ParamMaps, and an Evaluator.
tvs = TrainValidationSplit(
    estimator=rf,
    estimatorParamMaps=paramGrid,
    evaluator=BinaryClassificationEvaluator(),
    trainRatio=0.8,
)  # 80% of the data will be used for training, 20% for validation.

# Run TrainValidationSplit, and choose the best set of parameters.
model = tvs.fit(train)


In [ ]:
# From the best model take chosen hyper-parameters 
best_model = model.bestModel.extractParamMap()

for key in best_model.keys():
    print(str(key).split("_")[-1], ":", best_model[key])


In [ ]:
# Use defined function plot roc curve and print roc auc score
plot_roc_curve(model, train, test)


In [ ]:
# Evaluate the feature importances and plot the charts for most important 20 features
importance = model.bestModel.featureImportances
features_list = inputcols
feature_importance = pd.DataFrame(
    data={"features_list": features_list, "importance": importance}
)
feature_importance = feature_importance.sort_values(by="importance", ascending=False)[
    :20
]
sns.catplot(
    x="importance",
    y="features_list",
    data=feature_importance,
    kind="bar",
    height=5,
    aspect=2.7,
)
plt.ylabel("Feature", fontsize=15)
plt.xlabel("Feature Importance", fontsize=15)


### 5.3 Gradient-Boosted Trees(GBTs) Classifier

In [ ]:
# Validation for hyper-parameter tuning.
# Randomly splits the input dataset into train and validation sets,
# and uses evaluation metric on the validation set to select the best model.
gbt = GBTClassifier(
    featuresCol="features", labelCol="label", predictionCol="prediction"
)

# Use a ParamGridBuilder to construct a grid of parameters to search over.
# TrainValidationSplit will try all combinations of values and determine best model using the evaluator.
paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [8, 10])
    .addGrid(gbt.minInstancesPerNode, [5, 20, 50])
    .addGrid(gbt.maxIter, [10, 20])
    .build()
)

# In this case the estimator is BinaryClassificationEvaluator
# A TrainValidationSplit requires an Estimator, a set of Estimator ParamMaps, and an Evaluator.
tvs = TrainValidationSplit(
    estimator=gbt,
    estimatorParamMaps=paramGrid,
    evaluator=BinaryClassificationEvaluator(),
    trainRatio=0.8,
)  # 80% of the data will be used for training, 20% for validation.

# Run TrainValidationSplit, and choose the best set of parameters.
model = tvs.fit(train)

# Make predictions on test data. model is the model with combination of parameters that performed best.
model.transform(test).select("features", "label", "prediction").show(10)


In [ ]:
# From the best model take chosen hyper-parameters 
best_model = model.bestModel.extractParamMap()

for key in best_model.keys():
    print(str(key).split("_")[-1], ":", best_model[key])


In [ ]:
# Use defined function plot roc curve and print roc auc score
plot_roc_curve(model, train, test)


### 5.4 Use XGBoost Regressor Algorithm Build Revenue Loss Prediction Model

In [ ]:
# Transofrm scaledData into pandas Dataframe so we can use XGBoost and Keras
dataset = scaledData.toPandas()

# Create revenue loss column as label
dataset["revenue_loss"] = dataset.label * dataset.MonthlyRevenue

# Create the list of features
feature_list = [
    column
    for column in dataset.columns
    if column
    not in ["CustomerID", "label", "features", "revenue_loss", "scaledFeatures"]
]

# Create X, y, and split into train/test sets
X = dataset.loc[:, feature_list]
y = dataset.revenue_loss
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
# Use GridSearch method to figure the best hyper-parameters
# Parameters to search over with cross-validation
grid_params = [
    {
        "max_depth": [4, 5],
        "learning_rate": [0.1, 1],
        "n_estimators": [10, 100],
        "reg_lambda": [100, 10],
        "objective": ["reg:linear"],
    }
]

xgbr = GridSearchCV(xgb.XGBRegressor(), grid_params, cv=5, scoring="r2")
xgbr.fit(X_train, y_train)
y_pred = xgbr.predict(X_test)

print("Mean Absolute Error: %r\n" % metrics.mean_absolute_error(y_test, y_pred))
print("Best parameter values: %r\n" % xgbr.best_params_)


In [ ]:
# Plot a chart of how many of total revenue loss can be captured with our predicted highest risk group
temp = X_test.copy()
temp["revenue_loss"] = y_test
temp["y_pred"] = y_pred
n = 20
temp["y_pred_bin"] = pd.qcut(temp["y_pred"], n, duplicates="drop")
l = len(temp["y_pred_bin"].unique())
labels = [l + 1 - i for i in range(1, l + 1)]
temp["y_pred_bin"] = pd.qcut(temp["y_pred"], n, labels=labels, duplicates="drop")
index_amount = temp.index[(temp.y_pred_bin == 1)]
fig = plt.figure(figsize=(10, 5))
labels.sort()
ax = sns.barplot(
    x="y_pred_bin", y="revenue_loss", data=temp, estimator=np.sum, order=labels
)
plt.xlabel("Predicted Risk Level", fontsize=15)
plt.ylabel("revenue_loss", fontsize=15)
per = 100 * sum(temp.y_pred_bin == 1) / len(temp.y_pred_bin)
print(
    "Highest Risk Group(%.2f%% of total clients) can capture %.2f%% of total revenue_loss"
    % (
        per,
        100 * temp[(temp.y_pred_bin == 1)].revenue_loss.sum() / temp.revenue_loss.sum(),
    )
)


### 5.5 Use Keras deploy Nural networks to predict revenue loss

In [ ]:
a = 0.0005
drop = 0.2
n = X_train.shape[1]

model = Sequential(
    [
        Dense(
            int(n * 1.2),
            input_dim=n,
            kernel_regularizer=regularizers.l1_l2(l1=a, l2=a),
            activation="relu",
        ),
        Dropout(drop),
        Dense(
            int(n), kernel_regularizer=regularizers.l1_l2(l1=a, l2=a), activation="relu"
        ),
        Dropout(drop),
        Dense(
            int(n * 0.8),
            activation="relu",
            kernel_regularizer=regularizers.l1_l2(l1=a, l2=a),
        ),
        Dense(1),
    ]
)


In [ ]:
# Prints a summary representation of model
model.summary()


In [ ]:
# Configures the model for training and fit the training set
model.compile(loss="mean_squared_error", optimizer="adam", metrics=["mae", "accuracy"])
history = model.fit(
    X_train, y_train, validation_split=0.2, epochs=5, verbose=1, shuffle=True
)


In [ ]:
# plot loss metrics
ax1 = plt.plot(history.history["loss"])
ax2 = plt.plot(history.history["val_loss"])
plt.legend(["loss", "val_loss"])


In [ ]:
# Use model redict test set
y_pred = model.predict(X_test)
y_test = np.array(y_test)


In [ ]:
# Plot a chart of how many of total revenue loss can be captured with our predicted highest risk group
temp = X_test.copy()
temp["revenue_loss"] = y_test
temp["y_pred"] = y_pred
n = 20
temp["y_pred_bin"] = pd.qcut(temp["y_pred"], n, duplicates="drop")
l = len(temp["y_pred_bin"].unique())
labels = [l + 1 - i for i in range(1, l + 1)]
temp["y_pred_bin"] = pd.qcut(temp["y_pred"], n, labels=labels, duplicates="drop")
index_amount = temp.index[(temp.y_pred_bin == 1)]
fig = plt.figure(figsize=(10, 5))
labels.sort()
ax = sns.barplot(
    x="y_pred_bin", y="revenue_loss", data=temp, estimator=np.sum, order=labels
)
plt.xlabel("Predicted Risk Level", fontsize=15)
plt.ylabel("revenue_loss", fontsize=15)
per = 100 * sum(temp.y_pred_bin == 1) / len(temp.y_pred_bin)
print(
    "Highest Risk Group(%.2f%% of total clients) can capture %.2f%% of total revenue_loss"
    % (
        per,
        100 * temp[(temp.y_pred_bin == 1)].revenue_loss.sum() / temp.revenue_loss.sum(),
    )
)


## 6. Customer Segmentation  <a name="Segmentation"></a>

In this module, we'll use KMeans to build a Customer Segmentation model, and then we can compare the different distributions of each cluster.

Customer segmentation is the practice of dividing customers into groups of individuals that are similar in specific ways relevant to marketing, such as age, months in service, services usage and services spending.

Based on the customer segmentation, the telecommunications company can customize their marketing efforts for different cluster, and gain a deeper understanding of their customers' preferences in order to more accurately tailor marketing materials.


In [ ]:
# Trains a k-means model
kmeans = KMeans(featuresCol="scaledFeatures", k=4, initSteps=2, tol=0.0001, maxIter=20)
model = kmeans.fit(scaledData)


In [ ]:
# Make predictions
predictions = model.transform(scaledData)


In [ ]:
# Evaluate clustering by computing Silhouette score
evaluator = ClusteringEvaluator()

silhouette = evaluator.evaluate(predictions)
print("Silhouette with squared euclidean distance = " + str(silhouette))


In [ ]:
# Create a Pandas DataFrame for data visualization
cluster = predictions.toPandas()


In [ ]:
# Compare the different distributions of each cluster
temp_list = list(feature_importance.iloc[:5, 0])
temp_list.append("prediction")
temp = cluster.loc[:, temp_list]

fig, axs = plt.subplots(figsize=(15, 5), ncols=len(temp_list) - 1)

for i in range(len(temp_list) - 1):
    ax = sns.barplot(x="prediction", y=temp_list[i], data=temp, ax=axs[i])
    ax.set_xlabel("cluster", fontsize=12)
    ax.set_ylabel(temp_list[i], fontsize=12, color="blue")


## 7. Results and Discussion <a name="results"></a>

Due to the lack of missing some important feature, we can see the overall classification performance is not perfect enough. However, we can still catch the highest risk group so the stakeholders can focus on those groups based on the predicted risk level, so they can put less retention cost but save more revenue loss.

From the result of the segmentation model, we can see that all customers have been separated into different clusters perfectly. Customer segmentation begins with gathering and analyzing data and ends with acting on the information gathered in a way that is appropriate and effective. So the next step in real business is based on these clusters and distributions, following with analytical technique such as Uplift Modeling to models the difference in behavior between target and control groups, so the business owner can customize their up-sale and cross-sale strategy. 